In [21]:
"""
Full ML Experiment Package (Single-Stock Version Using AAPL)
-----------------------------------------------------------
This script is a simplified but complete version of the S&P500 package,
focused only on Apple (AAPL) to avoid Wikipedia scraping issues.

Features included:
1. Data download (Yahoo Finance)
2. Feature engineering (returns, volatility, RSI, MACD, ranges)
3. Train/test splits (time-series)
4. Models: Naive, Linear, Ridge, RandomForest, Logistic (for classification)
5. Metrics: RMSE, MAE, R², Direction Accuracy, AUC (classification)
6. Backtest: long-only based on predicted next-day return
7. Summary tables + CSV export

Author: ChatGPT (GPT-5)
"""

import os
import numpy as np
import pandas as pd
import yfinance as yf
from dataclasses import dataclass
from sklearn.linear_model import LinearRegression, Ridge, LogisticRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.metrics import accuracy_score, roc_auc_score, f1_score
from sklearn.model_selection import TimeSeriesSplit
import math

# =====================================================
# Config
# =====================================================
@dataclass
class Config:
    ticker: str = "AAPL"
    start: str = "2015-01-01"
    end: str = "2025-01-01"
    task: str = "regression"  # "regression" or "classification"
    n_splits: int = 6
    tx_cost_bps: float = 5.0
    prediction_thresh: float = 0.0

CFG = Config()

# =====================================================
# TA indicators
# =====================================================
def compute_rsi(series, period=14):
    # ensure input is Pandas Series
    if isinstance(series, np.ndarray):
        series = pd.Series(series)
    delta = series.diff()
    up = delta.clip(lower=0)
    down = -delta.clip(upper=0)
    ma_up = up.ewm(alpha=1/period, adjust=False).mean()
    ma_down = down.ewm(alpha=1/period, adjust=False).mean()
    return 100 - 100/(1 + ma_up/ma_down)

def compute_macd(series, fast=12, slow=26, signal=9):
    # ensure input is Pandas Series
    if isinstance(series, np.ndarray):
        series = pd.Series(series)
    fast_ema = series.ewm(span=fast, adjust=False).mean()
    slow_ema = series.ewm(span=slow, adjust=False).mean()
    macd = fast_ema - slow_ema
    signal_line = macd.ewm(span=signal, adjust=False).mean()
    hist = macd - signal_line
    return macd, signal_line, hist

# =====================================================
# Backtest (long-only)
# =====================================================
def backtest_long_only(pred, y_true, cost_bps=5.0):
    """
    Simple backtest: go long if predicted next-day return > 0.
    Accepts numpy arrays or pandas Series. Converts to Series to use .diff().
    """
    if isinstance(pred, np.ndarray):
        pred = pd.Series(pred)
    if isinstance(y_true, np.ndarray):
        # align index with pred if possible
        y_true = pd.Series(y_true, index=pred.index if isinstance(pred, pd.Series) else None)

    signal = (pred > 0).astype(int)
    gross_ret = signal * y_true

    # Costs on position changes
    trades = signal.diff().abs().fillna(0)
    cost = trades * (cost_bps/10000.0)

    net_ret = gross_ret - cost
    df_bt = pd.DataFrame({
        'signal': signal,
        'gross': gross_ret,
        'cost': -cost,
        'net': net_ret
    })

    def summarize(x):
        mean = x.mean()
        vol = x.std()
        sharpe = mean / (vol + 1e-12) * math.sqrt(252)
        curve = (1 + x).cumprod()
        peak = curve.cummax()
        mdd = (curve/peak - 1).min()
        cagr = curve.iloc[-1] ** (252/len(x)) - 1 if len(x) > 0 else np.nan
        return {'MeanDaily': mean, 'VolDaily': vol, 'Sharpe': sharpe, 'MaxDD': mdd, 'CAGR': cagr}

    s = {"net_" + k: v for k, v in summarize(df_bt['net']).items()}
    return df_bt, s

# =====================================================
# Evaluate metrics
# =====================================================
def eval_metrics_reg(y_true, y_pred):
    return {
        "RMSE": math.sqrt(mean_squared_error(y_true, y_pred)),
        "MAE": mean_absolute_error(y_true, y_pred),
        "R2": r2_score(y_true, y_pred),
        "DirectionalAcc": accuracy_score((y_true>0), (y_pred>0)),
    }

def eval_metrics_cls(y_true, probs):
    preds = (probs >= 0.5).astype(int)
    out = {"Acc": accuracy_score(y_true, preds),
           "F1": f1_score(y_true, preds)}
    try:
        out["AUC"] = roc_auc_score(y_true, probs)
    except:
        out["AUC"] = np.nan
    return out

# =====================================================
# Models
# =====================================================
def get_models(task):
    if task == "regression":
        return {
            "naive bayes": None,
            "linear regression": LinearRegression(),
            "ridge": Ridge(alpha=10.0),
            "random forest": RandomForestRegressor(n_estimators=300, max_depth=8, random_state=42)
        }
    else:
        return {
            "logit": LogisticRegression(max_iter=200),
            "random forest": RandomForestRegressor(n_estimators=300, max_depth=8, random_state=42)
        }

# =====================================================
# Main experiment
# =====================================================
def run_experiment(cfg: Config):
    # 1) Download data
    df = yf.download(cfg.ticker, start=cfg.start, end=cfg.end, auto_adjust=True)

    # 2) Features
    df["ret_1"] = df["Close"].pct_change()
    df["ret_5"] = df["Close"].pct_change(5)
    df["vol_21"] = df["ret_1"].rolling(21).std()
    df["range_hl"] = (df["High"] - df["Low"]) / df["Close"].shift(1)

    df["rsi"] = compute_rsi(df["Close"])
    df["macd"], df["macd_signal"], df["macd_hist"] = compute_macd(df["Close"])

    df["y_reg"] = df["Close"].pct_change(-1)
    df["y_cls"] = (df["y_reg"] > cfg.prediction_thresh).astype(int)

    df = df.dropna()

    # 3) Prepare data
    feat_cols = ["ret_1","ret_5","vol_21","range_hl","rsi","macd","macd_signal","macd_hist"]
    X = df[feat_cols].values
    y = df["y_reg"].values if cfg.task == "regression" else df["y_cls"].values

    # 4) Splits
    tscv = TimeSeriesSplit(n_splits=cfg.n_splits)
    models = get_models(cfg.task)

    out_rows = []
    pnl_rows = []

    # 5) For each model
    for name, model in models.items():
        fold_metrics = []

        for fold, (train_idx, test_idx) in enumerate(tscv.split(X)):
            X_train, X_test = X[train_idx], X[test_idx]
            y_train, y_test = y[train_idx], y[test_idx]

            # naive model
            if name == "naive bayes" and cfg.task == "regression":
                pred = np.full_like(y_test, fill_value=np.mean(y_train))
            else:
                model.fit(X_train, y_train)
                pred = model.predict(X_test)

            # metrics
            if cfg.task == "regression":
                m = eval_metrics_reg(y_test, pred)
            else:
                m = eval_metrics_cls(y_test, pred)

            # Backtest (use Series with aligned index)
            test_index = df.iloc[test_idx].index
            y_true_ret = pd.Series(df["y_reg"].iloc[test_idx].values, index=test_index)
            pred_series = pd.Series(pred, index=test_index)
            pnl, bt = backtest_long_only(pred_series, y_true_ret, cost_bps=cfg.tx_cost_bps)

            m.update(bt)
            m["model"] = name
            m["fold"] = fold
            fold_metrics.append(m)

        # aggregate
        out_rows += fold_metrics

    results = pd.DataFrame(out_rows)
    agg = results.groupby("model").mean().reset_index()

    # save
    os.makedirs("artifacts", exist_ok=True)
    agg.to_csv("artifacts/model_comparison_aapl.csv", index=False)

    return agg, results

# =====================================================
# Run
# =====================================================
if __name__ == "__main__":
    agg, full = run_experiment(CFG)
    print("\n=== Model Comparison Summary ===")
    print(agg.round(4))
    print("\nSaved to artifacts/model_comparison_aapl.csv")


[*********************100%***********************]  1 of 1 completed



=== Model Comparison Summary ===
               model    RMSE     MAE      R2  DirectionalAcc  net_MeanDaily  \
0  linear regression  0.0178  0.0127 -0.0349          0.5075        -0.0004   
1        naive bayes  0.0176  0.0126 -0.0076          0.5248        -0.0003   
2      random forest  0.0188  0.0134 -0.1606          0.5089        -0.0004   
3              ridge  0.0177  0.0126 -0.0169          0.5290        -0.0003   

   net_VolDaily  net_Sharpe  net_MaxDD  net_CAGR  fold  
0        0.0124     -0.6976    -0.2809   -0.1154   2.5  
1        0.0019     -0.3817    -0.0780   -0.0584   2.5  
2        0.0105     -0.7000    -0.2439   -0.1107   2.5  
3        0.0078     -0.4647    -0.1635   -0.0794   2.5  

Saved to artifacts/model_comparison_aapl.csv


In [2]:
CFG

Config(ticker='AAPL', start='2015-01-01', end='2025-01-01', task='regression', n_splits=6, tx_cost_bps=5.0, prediction_thresh=0.0)

In [25]:
df = yf.download(CFG.ticker, start=CFG.start, end=CFG.end, auto_adjust=True)

[*********************100%***********************]  1 of 1 completed


In [27]:
df

Price,Close,High,Low,Open,Volume
Ticker,AAPL,AAPL,AAPL,AAPL,AAPL
Date,,,,,
2015-01-02,24.261047,24.729270,23.821672,24.718174,212818400
2015-01-05,23.577574,24.110150,23.391173,24.030263,257142000
2015-01-06,23.579794,23.839424,23.218085,23.641928,263188400
2015-01-07,23.910433,24.010290,23.677430,23.788384,160423600
2015-01-08,24.829128,24.886824,24.121246,24.238858,237458000
...,...,...,...,...,...
2024-12-24,257.286682,257.296626,254.386957,254.586262,23234700
2024-12-26,258.103729,259.179926,256.718662,257.276679,27237100


In [26]:
df.describe()

Price,Close,High,Low,Open,Volume
Ticker,AAPL,AAPL,AAPL,AAPL,AAPL
count,2516.000000,2516.000000,2516.000000,2516.000000,2.516000e+03
mean,93.949921,94.862311,92.935270,93.863011,1.170853e+08
std,65.504728,66.098727,64.817392,65.424437,6.839614e+07
min,20.624052,20.927680,20.425438,20.546430,2.323470e+07
25%,35.257094,35.655406,34.878665,35.277318,7.105610e+07
50%,64.450447,65.005706,63.647693,64.290837,1.003646e+08
75%,150.379662,152.129378,148.411451,150.202255,1.426216e+08
max,258.103729,259.179926,256.718662,257.276679,6.488252e+08


In [7]:
df.tail()

Price,Close,High,Low,Open,Volume
Ticker,AAPL,AAPL,AAPL,AAPL,AAPL
Date,,,,,
2024-12-24,257.286682,257.296626,254.386957,254.586262,23234700
2024-12-26,258.103729,259.179926,256.718662,257.276679,27237100
2024-12-27,254.685867,257.784882,252.164818,256.917934,42355300
2024-12-30,251.307877,252.603281,249.863009,251.337769,35557500
2024-12-31,249.534180,252.384064,248.547676,251.547039,39480700


In [8]:
agg, full = run_experiment(CFG)

[*********************100%***********************]  1 of 1 completed


In [ ]:
# RSI indicator example:

def compute_rsi(series, period=14):
    if isinstance(series, np.ndarray):
        series = pd.Series(series)
    delta = series.diff()
    up = delta.clip(lower=0)
    down = -delta.clip(upper=0)
    ma_up = up.ewm(alpha=1/period, adjust=False).mean()
    ma_down = down.ewm(alpha=1/period, adjust=False).mean()
    return 100 - (100 / (1 + ma_up/ma_down))

rsi_periods = [5, 10, 14, 20, 60, 120]

# Compute and add RSI for each period
for period in rsi_periods:
    col_name = f"rsi_{period}"
    df[col_name] = compute_rsi(df["Close"], period=period)


# # Optional: Add RSI-based signals (e.g., overbought/oversold flags)
# # Example: Binary signal if RSI(14) > 70 (overbought) or < 30 (oversold)
# df["rsi_14_overbought"] = (df["rsi_14"] > 70).astype(int)
# df["rsi_14_oversold"]   = (df["rsi_14"] < 30).astype(int)

# # Optional: RSI rate-of-change (momentum of momentum)
# df["rsi_14_roc"] = df["rsi_14"].pct_change()

In [11]:
cols = ["rsi_5", "rsi_10", "rsi_14", "rsi_20", "rsi_60", "rsi_120"]
df[cols].tail()

Price,rsi_5,rsi_10,rsi_14,rsi_20,rsi_60,rsi_120
Ticker,,,,,,
Date,,,,,,
2024-12-24,80.580192,77.848047,75.750282,72.409437,62.499285,59.046018
2024-12-26,82.048724,78.723035,76.452858,72.976182,62.745841,59.180690
2024-12-27,58.798693,66.511934,67.626246,66.922965,61.038628,58.370998
2024-12-30,43.552050,56.831334,60.225593,61.606624,59.413851,57.585776
2024-12-31,37.217887,52.383244,56.715929,59.015426,58.581192,57.178505


In [16]:
agg

,model,RMSE,MAE,R2,DirectionalAcc,net_MeanDaily,net_VolDaily,net_Sharpe,net_MaxDD,net_CAGR,fold
0,linear,0.017805,0.012749,-0.034873,0.507491,-0.000446,0.012403,-0.697606,-0.280906,-0.115363,2.5
1,naive,0.017617,0.012553,-0.007612,0.524813,-0.000274,0.001899,-0.381658,-0.077991,-0.058366,2.5
2,rf,0.018832,0.013423,-0.160583,0.508895,-0.000437,0.010509,-0.700006,-0.243882,-0.110687,2.5
3,ridge,0.017681,0.012590,-0.016928,0.529026,-0.000312,0.007830,-0.464674,-0.163461,-0.079404,2.5


In [22]:
agg[["model", "net_MeanDaily", "net_VolDaily", "net_Sharpe", "net_MaxDD"]]

,model,net_MeanDaily,net_VolDaily,net_Sharpe,net_MaxDD
0,linear regression,-0.000446,0.012403,-0.697606,-0.280906
1,naive bayes,-0.000274,0.001899,-0.381658,-0.077991
2,random forest,-0.000437,0.010509,-0.700006,-0.243882
3,ridge,-0.000312,0.007830,-0.464674,-0.163461
